# Conversión de Modelos para Despliegue en ESP32-S3

## Notebook de preparación — 03_ING_DESPLIEGUE

**Objetivo:** Convertir los 3 modelos seleccionados en el Ciclo 1 de Entrenamiento a los formatos requeridos para despliegue en la Freenove ESP32-S3 CAM Board (WROOM N16R8 + OV5640).

### Modelos seleccionados

| Modelo | Familia | TFLite INT8 (MB) | mAP@50 TFLite | Output shape | NMS |
|---|---|---|---|---|---|
| **MBNTv2_ssdlite_v1** | MobileNetV2 | 1.20 | 0.387 | 3 tensores (bbox, class, obj) | Decode anchors + NMS |
| **YOLO11n_v1** | YOLO11 Nano | 2.68 | 0.806 | `[1,9,1029]` | NMS completo en device |
| **YOLO26n_v1** | YOLO26 Nano | 2.55 | 0.790 | `[1,300,6]` | NMS integrado |

### Runtimes de inferencia (dual-path)

1. **TFLite Micro** → `.tflite` → C header (`.h`) vía `xxd -i`
2. **ESP-DL** → `.onnx` → `.espdl` vía `esp-ppq` (cuantización INT8 simétrica)

### Flujo de conversión

```
                    ┌─────────────────────┐
                    │  Modelos originales  │
                    │  (.tflite, .onnx,    │
                    │   .keras, .pt)       │
                    └────────┬────────────┘
                             │
              ┌──────────────┼──────────────┐
              ▼                             ▼
     ┌─────────────────┐          ┌──────────────────┐
     │  Path TFLite     │          │  Path ESP-DL      │
     │  .tflite → .h    │          │  .onnx → .espdl   │
     │  (xxd -i)        │          │  (esp-ppq)         │
     └────────┬────────┘          └────────┬─────────┘
              │                             │
              ▼                             ▼
     main/models/tflite/           main/models/espdl/
```

---
# 1. Setup y verificación de modelos fuente

In [1]:
"""
1.1 — Imports y configuración de rutas
"""
from pathlib import Path
import struct
import json
import subprocess
import shutil
import gzip
import os

import numpy as np
import pandas as pd

# ─── Rutas base ──────────────────────────────────────────────────────────────
ROOT       = Path("/Users/admin/Documents/TFM_UNIR")
OUTPUTS    = ROOT / "02_ING_MODELOS/GoogleCloudAI/outputs"
DEPLOY     = ROOT / "03_ING_DESPLIEGUE"
MODELS_TFL = DEPLOY / "main/models/tflite"
MODELS_DL  = DEPLOY / "main/models/espdl"

# Crear directorios de destino
MODELS_TFL.mkdir(parents=True, exist_ok=True)
MODELS_DL.mkdir(parents=True, exist_ok=True)

# ─── Definición de modelos ───────────────────────────────────────────────────
MODELS = {
    "MBNTv2_ssdlite_v1": {
        "tflite": OUTPUTS / "MBNTv2_ssdlite_v1/tflite/MBNTv2_ssdlite_v1_int8.tflite",
        "keras":  OUTPUTS / "MBNTv2_ssdlite_v1/MBNTv2_ssdlite_v1_final.keras",
        "onnx":   None,  # Se exportará desde .keras en sección 3
        "header_name": "mobilenetv2_ssdlite_v1_int8",
        "espdl_name":  "mobilenetv2_ssdlite_v1",
        "family": "MobileNetV2",
        "output_desc": "3 tensores: bbox[1,1470,4], class[1,1470,5], obj[1,1470,1]",
        "nms": "Decode anchors + NMS",
    },
    "YOLO11n_v1": {
        "tflite": OUTPUTS / "yolo11n_v1/tflite/best_int8.tflite",
        "onnx":   OUTPUTS / "yolo11n_v1/train/weights/best.onnx",
        "header_name": "yolo11n_v1_int8",
        "espdl_name":  "yolo11n_v1",
        "family": "YOLO11",
        "output_desc": "1 tensor: [1,9,1029]",
        "nms": "NMS completo en device",
    },
    "YOLO26n_v1": {
        "tflite": OUTPUTS / "yolo26n_v1/tflite/best_int8.tflite",
        "onnx":   OUTPUTS / "yolo26n_v1/train/weights/best.onnx",
        "header_name": "yolo26n_v1_int8",
        "espdl_name":  "yolo26n_v1",
        "family": "YOLO26",
        "output_desc": "1 tensor: [1,300,6]",
        "nms": "NMS integrado (no requiere NMS)",
    },
}

print(f"Root:       {ROOT}")
print(f"Outputs:    {OUTPUTS}")
print(f"Deploy:     {DEPLOY}")
print(f"TFLite dir: {MODELS_TFL}")
print(f"ESPDL dir:  {MODELS_DL}")
print(f"\n✅ {len(MODELS)} modelos configurados: {list(MODELS.keys())}")

Root:       /Users/admin/Documents/TFM_UNIR
Outputs:    /Users/admin/Documents/TFM_UNIR/02_ING_MODELOS/GoogleCloudAI/outputs
Deploy:     /Users/admin/Documents/TFM_UNIR/03_ING_DESPLIEGUE
TFLite dir: /Users/admin/Documents/TFM_UNIR/03_ING_DESPLIEGUE/main/models/tflite
ESPDL dir:  /Users/admin/Documents/TFM_UNIR/03_ING_DESPLIEGUE/main/models/espdl

✅ 3 modelos configurados: ['MBNTv2_ssdlite_v1', 'YOLO11n_v1', 'YOLO26n_v1']


In [2]:
"""
1.2 — Verificación de archivos fuente y análisis de modelos TFLite
"""
import tensorflow as tf

rows = []
for name, cfg in MODELS.items():
    # ─── Verificar .tflite ──────────────────────────────────────────────
    tfl_path = cfg["tflite"]
    tfl_exists = tfl_path.exists()
    tfl_size = tfl_path.stat().st_size / (1024 * 1024) if tfl_exists else 0

    # Analizar model con TFLite interpreter
    input_shape, output_shapes, ops_set = "—", "—", set()
    if tfl_exists:
        interpreter = tf.lite.Interpreter(model_path=str(tfl_path))
        interpreter.allocate_tensors()

        inp = interpreter.get_input_details()
        out = interpreter.get_output_details()
        input_shape = str(inp[0]["shape"].tolist())
        output_shapes = " | ".join(str(o["shape"].tolist()) for o in out)

        # Extraer operadores usados (para MicroMutableOpResolver)
        # Leer del flatbuffer model
        try:
            model_content = tfl_path.read_bytes()
            model_obj = tf.lite.experimental.Analyzer.analyze(
                model_content=model_content, gpu_compatibility=False
            )
        except Exception:
            pass  # Análisis de ops se hará abajo con API alternativa

    # ─── Verificar .onnx ────────────────────────────────────────────────
    onnx_path = cfg.get("onnx")
    onnx_exists = onnx_path.exists() if onnx_path else False
    onnx_size = onnx_path.stat().st_size / (1024 * 1024) if onnx_exists else 0

    # ─── Verificar .keras ───────────────────────────────────────────────
    keras_path = cfg.get("keras")
    keras_exists = keras_path.exists() if keras_path else False

    rows.append({
        "Modelo": name,
        "Familia": cfg["family"],
        "TFLite": "✅" if tfl_exists else "❌",
        "TFLite (MB)": f"{tfl_size:.2f}",
        "Input shape": input_shape,
        "Output shapes": output_shapes,
        "N° outputs": len(out) if tfl_exists else 0,
        "ONNX": "✅" if onnx_exists else ("⏳ exportar" if keras_exists else "❌"),
        "ONNX (MB)": f"{onnx_size:.2f}" if onnx_exists else "—",
        "Keras": "✅" if keras_exists else "—",
        "NMS": cfg["nms"],
    })

df_check = pd.DataFrame(rows).set_index("Modelo")
display(df_check)

# ─── Resumen de estado ──────────────────────────────────────────────────
all_tfl = all(r["TFLite"] == "✅" for r in rows)
all_onnx = all(r["ONNX"] == "✅" for r in rows)
print(f"\n{'✅' if all_tfl else '⚠️'} TFLite INT8: {'todos disponibles' if all_tfl else 'faltan archivos'}")
print(f"{'✅' if all_onnx else '⏳'} ONNX: {'todos disponibles' if all_onnx else 'MBNTv2 requiere exportación desde .keras'}")

=== TFLite ModelAnalyzer ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the CONV_2D op takes
tensor #0 and tensor #125 and tensor #124 as input and produces tensor #126 as output.

Subgraph#0 main(T#0) -> [T#216, T#209, T#202]
  Op#0 CONV_2D(T#0, T#125, T#124[-174275, -1073741824, -20696, 1073741824, 198754, ...]) -> [T#126]
  Op#1 DEPTHWISE_CONV_2D(T#126, T#123, T#122[9601, -151, 17408, -1023, 15223, ...]) -> [T#127]
  Op#2 CONV_2D(T#127, T#121, T#120[-9610, -5789, -7957, -29658, 29309, ...]) -> [T#128]
  Op#3 CONV_2D(T#128, T#119, T#118[6140, 1754, 436, 9, 56, ...]) -> [T#129]
  Op#4 PAD(T#129, T#1[0, 0, 0, 1, 0, ...]) -> [T#130]
  Op#5 DEPTHWISE_CONV_2D(T#130, T#117, T#116[10682, -830, 88264, 31804, 87422, ...]) -> [T#131]
  Op#6 CONV_2D(T#131, T#115, T#114[32580, 16297, -723, -19271, -13417, ...]) -> [T#132]
  Op#7 CONV_2D(T#132, T#113, T#112[5029, -712, -3278, 3187, 5533, ...]) -> [T#133]

INFO: Initialized TensorFlow Lite runtime.
INFO: Applying 1 TensorFlow Lite delegate(s) lazily.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
VERBOSE: XNNPack weight cache not enabled.
VERBOSE: Replacing 79 out of 91 node(s) with delegate (TfLiteXNNPackDelegate) node, yielding 3 partitions for the whole graph.
INFO: Successfully applied the default TensorFlow Lite delegate indexed at 0.
 *NOTE*: because a delegate has been applied, the precision of computations should be unchanged, but the exact output tensor values may have changed. If such output values are checked in your code, like in your tests etc., please consider increasing error tolerance for the check.
INFO: Applying 1 TensorFlow Lite delegate(s) lazily.
VERBOSE: XNNPack weight cache not enabled.
VERBOSE: Replacing 315 out of 343 node(s) with delegate (TfLiteXNNPackDelegate) node, yielding 27 partitions for the whole graph.
INFO: Successfully applied the default TensorFlow Lite delegate indexed at 0.
 *NOTE*: becaus

,Familia,TFLite,TFLite (MB),Input shape,Output shapes,N° outputs,ONNX,ONNX (MB),Keras,NMS
Modelo,,,,,,,,,,
MBNTv2_ssdlite_v1,MobileNetV2,✅,1.20,"[1, 224, 224, 3]","[1, 1470, 1] | [1, 1470, 5] | [1, 1470, 4]",3,⏳ exportar,—,✅,Decode anchors + NMS
YOLO11n_v1,YOLO11,✅,2.68,"[1, 224, 224, 3]","[1, 9, 1029]",1,✅,9.97,—,NMS completo en device
YOLO26n_v1,YOLO26,✅,2.55,"[1, 224, 224, 3]","[1, 300, 6]",1,✅,9.20,—,NMS integrado (no requiere NMS)



✅ TFLite INT8: todos disponibles
⏳ ONNX: MBNTv2 requiere exportación desde .keras


In [3]:
"""
1.3 — Análisis de operadores TFLite (para MicroMutableOpResolver)

Extraemos los operadores únicos de cada modelo para configurar
el OpResolver mínimo en el firmware — evitando AllOpsResolver
que consume ~100 KB adicionales de flash.
"""

def get_tflite_ops(tflite_path: Path) -> list[str]:
    """Extrae la lista de operadores únicos de un modelo TFLite."""
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    # Usar la API _get_ops_details() del interpreter
    try:
        ops_details = interpreter._get_ops_details()
        ops = sorted(set(op["op_name"] for op in ops_details))
        return ops
    except AttributeError:
        # Fallback: leer desde el flatbuffer directamente
        return ["(análisis no disponible con esta versión de TF)"]


all_ops = set()
ops_report = {}

for name, cfg in MODELS.items():
    ops = get_tflite_ops(cfg["tflite"])
    ops_report[name] = ops
    all_ops.update(ops)
    print(f"📦 {name}: {len(ops)} operadores")
    for op in ops:
        print(f"   └─ {op}")
    print()

# Operadores comunes a todos los modelos
common_ops = set.intersection(*[set(ops) for ops in ops_report.values()])
print(f"🔗 Operadores comunes ({len(common_ops)}): {sorted(common_ops)}")
print(f"📋 Total operadores únicos ({len(all_ops)}): {sorted(all_ops)}")
print(f"\n💡 El MicroMutableOpResolver necesita registrar {len(all_ops)} operadores para soportar los 3 modelos.")

📦 MBNTv2_ssdlite_v1: 10 operadores
   └─ ADD
   └─ CONV_2D
   └─ DEPTHWISE_CONV_2D
   └─ DEQUANTIZE
   └─ LOGISTIC
   └─ PACK
   └─ PAD
   └─ RESHAPE
   └─ SHAPE
   └─ STRIDED_SLICE

📦 YOLO11n_v1: 15 operadores
   └─ ADD
   └─ BATCH_MATMUL
   └─ CONCATENATION
   └─ CONV_2D
   └─ DEPTHWISE_CONV_2D
   └─ LOGISTIC
   └─ MAX_POOL_2D
   └─ MUL
   └─ PAD
   └─ RESHAPE
   └─ RESIZE_NEAREST_NEIGHBOR
   └─ SOFTMAX
   └─ STRIDED_SLICE
   └─ SUB
   └─ TRANSPOSE

📦 YOLO26n_v1: 23 operadores
   └─ ADD
   └─ BATCH_MATMUL
   └─ CAST
   └─ CONCATENATION
   └─ CONV_2D
   └─ DEPTHWISE_CONV_2D
   └─ FLOOR_MOD
   └─ GATHER
   └─ GATHER_ND
   └─ LESS
   └─ LOGISTIC
   └─ MAX_POOL_2D
   └─ MUL
   └─ PAD
   └─ REDUCE_MAX
   └─ RESHAPE
   └─ RESIZE_NEAREST_NEIGHBOR
   └─ SOFTMAX
   └─ STRIDED_SLICE
   └─ SUB
   └─ TILE
   └─ TOPK_V2
   └─ TRANSPOSE

🔗 Operadores comunes (7): ['ADD', 'CONV_2D', 'DEPTHWISE_CONV_2D', 'LOGISTIC', 'PAD', 'RESHAPE', 'STRIDED_SLICE']
📋 Total operadores únicos (26): ['ADD', 'BATCH_MA

---
# 2. Path TFLite Micro: conversión `.tflite` → C header (`.h`)

Convertimos cada archivo `.tflite` a un array C embebible en flash.
Se usa un equivalente Python de `xxd -i` para generar los headers.

Cada header generado contiene:
```c
// Auto-generated — do not edit
alignas(16) const unsigned char <name>_data[] = { 0x.., 0x.., ... };
const unsigned int <name>_data_len = <size>;
```

Los arrays se alinean a 16 bytes para acceso óptimo en Xtensa LX7.

In [4]:
"""
2.1 — Función de conversión TFLite → C header
"""

def tflite_to_c_header(
    tflite_path: Path,
    output_path: Path,
    var_name: str,
    bytes_per_line: int = 12,
) -> dict:
    """
    Convierte un .tflite a un C header embebible.
    
    Args:
        tflite_path: Ruta al archivo .tflite
        output_path: Ruta de salida del .h
        var_name: Nombre base de la variable C (sin sufijo _data)
        bytes_per_line: Bytes por línea en el array C
    
    Returns:
        dict con estadísticas de la conversión
    """
    data = tflite_path.read_bytes()
    size = len(data)
    
    # Generar el array C
    lines = []
    lines.append(f"// Auto-generated from {tflite_path.name}")
    lines.append(f"// Model size: {size:,} bytes ({size / (1024*1024):.2f} MB)")
    lines.append(f"// DO NOT EDIT — regenerate with Conversion_ModelosTFLite.ipynb")
    lines.append(f"")
    lines.append(f"#ifndef {var_name.upper()}_H")
    lines.append(f"#define {var_name.upper()}_H")
    lines.append(f"")
    lines.append(f"#include <cstdint>")
    lines.append(f"")
    lines.append(f"alignas(16) const unsigned char {var_name}_data[] = {{")
    
    # Formatear bytes en filas
    for i in range(0, size, bytes_per_line):
        chunk = data[i:i + bytes_per_line]
        hex_vals = ", ".join(f"0x{b:02x}" for b in chunk)
        comma = "," if i + bytes_per_line < size else ""
        lines.append(f"    {hex_vals}{comma}")
    
    lines.append(f"}};")
    lines.append(f"")
    lines.append(f"const unsigned int {var_name}_data_len = {size};")
    lines.append(f"")
    lines.append(f"#endif // {var_name.upper()}_H")
    lines.append(f"")
    
    header_content = "\n".join(lines)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(header_content)
    
    header_size = len(header_content)
    
    return {
        "tflite_bytes": size,
        "tflite_mb": size / (1024 * 1024),
        "header_bytes": header_size,
        "header_mb": header_size / (1024 * 1024),
        "expansion_ratio": header_size / size,
        "output_path": str(output_path),
    }

print("✅ Función tflite_to_c_header() definida")

✅ Función tflite_to_c_header() definida


In [5]:
"""
2.2 — Convertir los 3 modelos TFLite a C headers
"""

conversion_results = []

for name, cfg in MODELS.items():
    header_path = MODELS_TFL / f"{cfg['header_name']}.h"
    
    print(f"🔄 Convirtiendo {name}...")
    result = tflite_to_c_header(
        tflite_path=cfg["tflite"],
        output_path=header_path,
        var_name=cfg["header_name"],
    )
    result["model"] = name
    conversion_results.append(result)
    print(f"   ✅ → {header_path.name} ({result['header_mb']:.2f} MB, ratio ×{result['expansion_ratio']:.1f})")

print()

# Tabla resumen
df_conv = pd.DataFrame(conversion_results).set_index("model")
df_conv_display = df_conv[["tflite_mb", "header_mb", "expansion_ratio"]].copy()
df_conv_display.columns = ["TFLite (MB)", "Header (MB)", "Ratio expansión"]
display(df_conv_display)

total_tfl = df_conv["tflite_mb"].sum()
total_hdr = df_conv["header_mb"].sum()
print(f"\n📊 Total TFLite: {total_tfl:.2f} MB")
print(f"📊 Total headers: {total_hdr:.2f} MB")
print(f"📊 Los 3 modelos caben en la partición factory de 15 MB: {'✅ Sí' if total_tfl < 15 else '❌ No'}")
print(f"\n📁 Headers generados en: {MODELS_TFL}")

🔄 Convirtiendo MBNTv2_ssdlite_v1...
   ✅ → mobilenetv2_ssdlite_v1_int8.h (7.61 MB, ratio ×6.3)
🔄 Convirtiendo YOLO11n_v1...
   ✅ → yolo11n_v1_int8.h (16.99 MB, ratio ×6.3)
🔄 Convirtiendo YOLO26n_v1...
   ✅ → yolo26n_v1_int8.h (16.17 MB, ratio ×6.3)



,TFLite (MB),Header (MB),Ratio expansión
model,,,
MBNTv2_ssdlite_v1,1.202278,7.614847,6.333682
YOLO11n_v1,2.682261,16.988000,6.333464
YOLO26n_v1,2.553178,16.170478,6.333471



📊 Total TFLite: 6.44 MB
📊 Total headers: 40.77 MB
📊 Los 3 modelos caben en la partición factory de 15 MB: ✅ Sí

📁 Headers generados en: /Users/admin/Documents/TFM_UNIR/03_ING_DESPLIEGUE/main/models/tflite


---
# 3. Path ESP-DL: conversión ONNX → `.espdl`

ESP-DL es el framework de inferencia nativo de Espressif, optimizado para ESP32-S3 con:
- **Scheduling dual-core automático** (Conv2D, DepthwiseConv2D)
- **Operadores SIMD en assembly nativo** (Xtensa LX7)
- **Cuantización INT8 simétrica** (POWER_OF_TWO) vía `esp-ppq`
- **Soporte documentado** para YOLO11n y MobileNetV2

### Pipeline de conversión

```
MBNTv2:  .keras → .onnx (tf2onnx) → .espdl (esp-ppq)
YOLO11n: .onnx  ──────────────────→ .espdl (esp-ppq)
YOLO26n: .onnx  ──────────────────→ .espdl (esp-ppq)
```

### Requisitos
- `pip install esp-ppq` — cuantizador oficial de Espressif (basado en `ppq`)
- `pip install tf2onnx` — para exportar MBNTv2 de Keras a ONNX  
- Dataset representativo para calibración de cuantización

> **⚠️ Nota:** La conversión ESP-DL requiere un dataset de calibración.
> Si `esp-ppq` no está disponible en este entorno, se genera un script
> de conversión standalone para ejecutar en un entorno con `esp-ppq`.

In [6]:
"""
3.1 — Exportar MBNTv2_ssdlite_v1 de Keras → ONNX (si no existe)
"""

# Verificar si tf2onnx está disponible
try:
    import tf2onnx
    HAS_TF2ONNX = True
    print(f"✅ tf2onnx v{tf2onnx.__version__} disponible")
except ImportError:
    HAS_TF2ONNX = False
    print("⚠️ tf2onnx no instalado — se instalará a continuación")

if not HAS_TF2ONNX:
    import subprocess
    subprocess.check_call(["pip", "install", "tf2onnx", "-q"])
    import tf2onnx
    print(f"✅ tf2onnx v{tf2onnx.__version__} instalado correctamente")

⚠️ tf2onnx no instalado — se instalará a continuación
✅ tf2onnx v1.16.1 instalado correctamente


In [8]:
"""
3.2 — Convertir MBNTv2_ssdlite_v1 Keras → ONNX

Nota: El modelo Keras fue compilado con losses custom (closures 'fn')
que no están registradas con @keras.saving.register_keras_serializable().
Se carga con compile=False ya que solo necesitamos la arquitectura + pesos
para la exportación a ONNX.
"""
import onnx

mbntv2_keras_path = MODELS["MBNTv2_ssdlite_v1"]["keras"]
mbntv2_onnx_path  = OUTPUTS / "MBNTv2_ssdlite_v1/MBNTv2_ssdlite_v1.onnx"

if mbntv2_onnx_path.exists():
    print(f"✅ ONNX ya existe: {mbntv2_onnx_path}")
    print(f"   Tamaño: {mbntv2_onnx_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print(f"🔄 Cargando modelo Keras (compile=False): {mbntv2_keras_path.name}...")
    model = tf.keras.models.load_model(str(mbntv2_keras_path), compile=False)
    
    print(f"   Input shape:  {model.input_shape}")
    print(f"   Output names: {[o.name for o in model.outputs]}")
    print(f"   Output shapes: {[o.shape for o in model.outputs]}")
    
    print(f"🔄 Convirtiendo a ONNX (opset 13)...")
    import tf2onnx
    onnx_model, _ = tf2onnx.convert.from_keras(
        model,
        input_signature=[tf.TensorSpec(shape=(1, 224, 224, 3), dtype=tf.float32, name="input")],
        opset=13,
        output_path=str(mbntv2_onnx_path),
    )
    
    print(f"✅ ONNX exportado: {mbntv2_onnx_path}")
    print(f"   Tamaño: {mbntv2_onnx_path.stat().st_size / (1024*1024):.2f} MB")
    
    del model  # Liberar memoria

# Actualizar referencia
MODELS["MBNTv2_ssdlite_v1"]["onnx"] = mbntv2_onnx_path

# Verificar todos los ONNX
print("\n📋 Estado de archivos ONNX:")
for name, cfg in MODELS.items():
    onnx_p = cfg.get("onnx")
    if onnx_p and onnx_p.exists():
        print(f"   ✅ {name}: {onnx_p.name} ({onnx_p.stat().st_size / (1024*1024):.2f} MB)")
    else:
        print(f"   ❌ {name}: no disponible")

🔄 Cargando modelo Keras (compile=False): MBNTv2_ssdlite_v1_final.keras...
   Input shape:  (None, 224, 224, 3)
   Output names: ['keras_tensor_715', 'keras_tensor_717', 'keras_tensor_719']
   Output shapes: [(None, 1470, 4), (None, 1470, 5), (None, 1470, 1)]
🔄 Convirtiendo a ONNX (opset 13)...


I0000 00:00:1770838543.805540  327022 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1770838543.807490  327022 single_machine.cc:361] Starting new session
I0000 00:00:1770838544.402163  327022 devices.cc:76] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0 (Note: TensorFlow was not compiled with CUDA or ROCm support)
I0000 00:00:1770838544.402246  327022 single_machine.cc:361] Starting new session
rewriter <function rewrite_constant_fold at 0x3084a13f0>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.


✅ ONNX exportado: /Users/admin/Documents/TFM_UNIR/02_ING_MODELOS/GoogleCloudAI/outputs/MBNTv2_ssdlite_v1/MBNTv2_ssdlite_v1.onnx
   Tamaño: 3.54 MB

📋 Estado de archivos ONNX:
   ✅ MBNTv2_ssdlite_v1: MBNTv2_ssdlite_v1.onnx (3.54 MB)
   ✅ YOLO11n_v1: best.onnx (9.97 MB)
   ✅ YOLO26n_v1: best.onnx (9.20 MB)


In [9]:
"""
3.3 — Conversión ONNX → ESPDL vía esp-ppq

esp-ppq es el cuantizador oficial de Espressif para generar modelos .espdl.
Requiere un dataset de calibración para la cuantización INT8 simétrica.

Si esp-ppq no está disponible en el entorno actual (macOS), se genera un
script standalone para ejecutar en un entorno compatible o directamente
en el ESP-IDF build environment.
"""

# Verificar disponibilidad de esp-ppq
try:
    import esp_ppq
    HAS_ESPPQ = True
    print(f"✅ esp-ppq disponible: v{esp_ppq.__version__}")
except ImportError:
    HAS_ESPPQ = False
    print("⚠️ esp-ppq no está disponible en este entorno")
    print("   Esto es normal en macOS — esp-ppq requiere entorno Linux/x86_64")
    print("   Se generará un script de conversión standalone")

# Intentar instalar si no está disponible
if not HAS_ESPPQ:
    try:
        import subprocess
        result = subprocess.run(
            ["pip", "install", "esp-ppq", "-q"],
            capture_output=True, text=True, timeout=30,
        )
        if result.returncode == 0:
            import esp_ppq
            HAS_ESPPQ = True
            print(f"\n✅ esp-ppq instalado: v{esp_ppq.__version__}")
        else:
            print(f"\n⚠️ No se pudo instalar esp-ppq: {result.stderr[:200]}")
    except Exception as e:
        print(f"\n⚠️ Error intentando instalar esp-ppq: {e}")

print(f"\n{'='*60}")
print(f"Estado: {'esp-ppq disponible → conversión directa' if HAS_ESPPQ else 'Generando script de conversión standalone'}")
print(f"{'='*60}")

⚠️ esp-ppq no está disponible en este entorno
   Esto es normal en macOS — esp-ppq requiere entorno Linux/x86_64
   Se generará un script de conversión standalone

⚠️ Error intentando instalar esp-ppq: No module named 'torch'

Estado: Generando script de conversión standalone


In [10]:
"""
3.4 — Generar script de conversión ONNX → ESPDL

Este script se puede ejecutar en un entorno con esp-ppq instalado
(requiere PyTorch + esp-ppq). Se recomienda usar Google Colab o
un entorno Linux con GPU para la calibración.
"""

convert_script = '''#!/usr/bin/env python3
"""
Script de conversión ONNX → ESPDL para ESP32-S3
Generado automáticamente por Conversion_ModelosTFLite.ipynb

Requisitos:
    pip install esp-ppq onnx numpy pillow

Uso:
    python convert_onnx_to_espdl.py [--calib-dir <path>] [--target esp32s3]
"""

import argparse
import os
import sys
from pathlib import Path

import numpy as np
import onnx

try:
    from esp_ppq import *
    from esp_ppq.api import espdl_quantize_onnx
except ImportError:
    print("ERROR: esp-ppq no instalado. Ejecutar: pip install esp-ppq")
    sys.exit(1)


# ─── Configuración de modelos ────────────────────────────────────────────
MODELS = {
    "mobilenetv2_ssdlite_v1": {
        "onnx": "02_ING_MODELOS/GoogleCloudAI/outputs/MBNTv2_ssdlite_v1/MBNTv2_ssdlite_v1.onnx",
        "input_shape": [1, 224, 224, 3],  # NHWC (TF convention)
        "channel_format": "nhwc",
    },
    "yolo11n_v1": {
        "onnx": "02_ING_MODELOS/GoogleCloudAI/outputs/yolo11n_v1/train/weights/best.onnx",
        "input_shape": [1, 3, 224, 224],  # NCHW (PyTorch/ONNX convention)
        "channel_format": "nchw",
    },
    "yolo26n_v1": {
        "onnx": "02_ING_MODELOS/GoogleCloudAI/outputs/yolo26n_v1/train/weights/best.onnx",
        "input_shape": [1, 3, 224, 224],  # NCHW
        "channel_format": "nchw",
    },
}

OUTPUT_DIR = Path("03_ING_DESPLIEGUE/main/models/espdl")


def create_calibration_dataset(calib_dir: str, input_shape: list, 
                                channel_format: str, n_samples: int = 64):
    """
    Genera un dataset de calibración a partir de imágenes.
    
    Si calib_dir está vacío o no existe, genera datos aleatorios (no ideal
    pero funcional para verificar la pipeline).
    """
    from PIL import Image
    
    h, w = 224, 224
    samples = []
    
    calib_path = Path(calib_dir) if calib_dir else None
    
    if calib_path and calib_path.exists():
        image_files = sorted(calib_path.glob("*.jpg")) + sorted(calib_path.glob("*.png"))
        image_files = image_files[:n_samples]
        print(f"  Usando {len(image_files)} imágenes de calibración de {calib_path}")
        
        for img_path in image_files:
            img = Image.open(img_path).convert("RGB").resize((w, h))
            arr = np.array(img, dtype=np.float32) / 255.0
            
            if channel_format == "nchw":
                arr = arr.transpose(2, 0, 1)  # HWC → CHW
            
            samples.append(np.expand_dims(arr, 0))
    else:
        print(f"  ⚠️ Sin directorio de calibración — usando datos aleatorios ({n_samples} muestras)")
        for _ in range(n_samples):
            if channel_format == "nchw":
                samples.append(np.random.rand(1, 3, h, w).astype(np.float32))
            else:
                samples.append(np.random.rand(1, h, w, 3).astype(np.float32))
    
    return samples


def convert_model(name: str, config: dict, calib_dir: str, target: str = "esp32s3"):
    """Convierte un modelo ONNX a formato ESPDL."""
    onnx_path = Path(config["onnx"])
    
    if not onnx_path.exists():
        print(f"  ❌ ONNX no encontrado: {onnx_path}")
        return False
    
    print(f"\\n{'='*60}")
    print(f"Convirtiendo: {name}")
    print(f"  ONNX: {onnx_path} ({onnx_path.stat().st_size / (1024*1024):.2f} MB)")
    print(f"  Input shape: {config['input_shape']}")
    print(f"  Target: {target}")
    
    # Generar dataset de calibración
    calib_data = create_calibration_dataset(
        calib_dir, config["input_shape"], config["channel_format"]
    )
    
    # Crear directorio de salida
    output_path = OUTPUT_DIR / name
    output_path.mkdir(parents=True, exist_ok=True)
    
    try:
        # Cuantización y exportación con esp-ppq
        quant_setting = QuantizationSettingFactory.espdl_setting()
        
        # Cuantización INT8 simétrica (estándar para ESP32-S3)
        ppq_graph = espdl_quantize_onnx(
            onnx_import_file=str(onnx_path),
            espdl_export_file=str(output_path / f"{name}.espdl"),
            calib_dataloader=calib_data,
            calib_steps=min(len(calib_data), 32),
            input_shape=config["input_shape"],
            target=target,
            setting=quant_setting,
            do_quantize=True,
        )
        
        espdl_file = output_path / f"{name}.espdl"
        if espdl_file.exists():
            print(f"  ✅ ESPDL generado: {espdl_file}")
            print(f"     Tamaño: {espdl_file.stat().st_size / (1024*1024):.2f} MB")
            return True
        else:
            print(f"  ❌ Error: archivo ESPDL no generado")
            return False
            
    except Exception as e:
        print(f"  ❌ Error en conversión: {e}")
        import traceback
        traceback.print_exc()
        return False


def main():
    parser = argparse.ArgumentParser(description="Convertir ONNX → ESPDL para ESP32-S3")
    parser.add_argument("--calib-dir", type=str, default="",
                        help="Directorio con imágenes de calibración (jpg/png)")
    parser.add_argument("--target", type=str, default="esp32s3",
                        choices=["esp32", "esp32s3", "esp32p4"],
                        help="Target de Espressif")
    parser.add_argument("--models", nargs="*", default=None,
                        help="Modelos a convertir (default: todos)")
    args = parser.parse_args()
    
    # Cambiar al directorio raíz del proyecto
    script_dir = Path(__file__).resolve().parent
    root = script_dir.parent if script_dir.name == "03_ING_DESPLIEGUE" else script_dir
    os.chdir(root)
    
    print(f"Directorio de trabajo: {os.getcwd()}")
    print(f"Target: {args.target}")
    print(f"Calibración: {args.calib_dir or '(datos aleatorios)'}")
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    models_to_convert = args.models or list(MODELS.keys())
    results = {}
    
    for name in models_to_convert:
        if name not in MODELS:
            print(f"⚠️ Modelo '{name}' no reconocido. Disponibles: {list(MODELS.keys())}")
            continue
        results[name] = convert_model(name, MODELS[name], args.calib_dir, args.target)
    
    # Resumen
    print(f"\\n{'='*60}")
    print("RESUMEN DE CONVERSIÓN")
    print(f"{'='*60}")
    for name, success in results.items():
        print(f"  {'✅' if success else '❌'} {name}")


if __name__ == "__main__":
    main()
'''

# Guardar script
script_path = DEPLOY / "convert_onnx_to_espdl.py"
script_path.write_text(convert_script)
print(f"✅ Script de conversión generado: {script_path}")
print(f"   Tamaño: {script_path.stat().st_size / 1024:.1f} KB")
print()
print("📋 Para ejecutar (en entorno con esp-ppq + PyTorch):")
print(f"   cd {ROOT}")
print(f"   python 03_ING_DESPLIEGUE/convert_onnx_to_espdl.py \\")
print(f"       --calib-dir 01_ING_DATOS/Datasets/dataset_maestro_unificado/data/ \\")
print(f"       --target esp32s3")
print()
print("📋 Alternativa — instalar esp-ppq en este entorno:")
print("   pip install torch esp-ppq")
print("   Luego re-ejecutar esta celda")

✅ Script de conversión generado: /Users/admin/Documents/TFM_UNIR/03_ING_DESPLIEGUE/convert_onnx_to_espdl.py
   Tamaño: 6.4 KB

📋 Para ejecutar (en entorno con esp-ppq + PyTorch):
   cd /Users/admin/Documents/TFM_UNIR
   python 03_ING_DESPLIEGUE/convert_onnx_to_espdl.py \
       --calib-dir 01_ING_DATOS/Datasets/dataset_maestro_unificado/data/ \
       --target esp32s3

📋 Alternativa — instalar esp-ppq en este entorno:
   pip install torch esp-ppq
   Luego re-ejecutar esta celda


---
# 4. Validación cruzada

Verificamos que la inferencia TFLite en Python produce resultados coherentes
con los resultados documentados en el Informe del Ciclo 1.

In [11]:
"""
4.1 — Validación de inferencia TFLite: verificar shapes de salida y rango de valores

Ejecutamos inferencia con una imagen sintética para validar que:
1. Los shapes de salida coinciden con los documentados
2. Los valores de salida están en rangos esperados
3. Los headers C generados producen el mismo binario que los .tflite originales
"""

def validate_tflite_model(tflite_path: Path, name: str) -> dict:
    """Ejecuta inferencia de prueba y reporta shapes/rangos."""
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    
    inp_detail = interpreter.get_input_details()[0]
    out_details = interpreter.get_output_details()
    
    # Generar input de prueba (imagen gris normalizada a INT8)
    input_shape = inp_detail["shape"]
    input_dtype = inp_detail["dtype"]
    
    if input_dtype == np.int8:
        test_input = np.random.randint(-128, 127, size=input_shape, dtype=np.int8)
    elif input_dtype == np.uint8:
        test_input = np.random.randint(0, 255, size=input_shape, dtype=np.uint8)
    else:
        test_input = np.random.rand(*input_shape).astype(np.float32)
    
    interpreter.set_tensor(inp_detail["index"], test_input)
    interpreter.invoke()
    
    outputs = {}
    for out in out_details:
        tensor = interpreter.get_tensor(out["index"])
        outputs[out["name"]] = {
            "shape": tensor.shape,
            "dtype": str(tensor.dtype),
            "min": float(tensor.min()),
            "max": float(tensor.max()),
            "mean": float(tensor.mean()),
            "quant_params": out.get("quantization_parameters", {}),
        }
    
    return {
        "name": name,
        "input_shape": input_shape.tolist(),
        "input_dtype": str(input_dtype),
        "n_outputs": len(out_details),
        "outputs": outputs,
    }


# Validar cada modelo
for name, cfg in MODELS.items():
    result = validate_tflite_model(cfg["tflite"], name)
    
    print(f"\n{'='*60}")
    print(f"📦 {name}")
    print(f"   Input:  {result['input_shape']} ({result['input_dtype']})")
    print(f"   Outputs: {result['n_outputs']}")
    
    for out_name, out_info in result["outputs"].items():
        print(f"   ├─ {out_name}")
        print(f"   │  Shape: {out_info['shape']}")
        print(f"   │  Dtype: {out_info['dtype']}")
        print(f"   │  Range: [{out_info['min']:.4f}, {out_info['max']:.4f}]")
        print(f"   │  Mean:  {out_info['mean']:.4f}")

# Validar integridad de headers C
print(f"\n{'='*60}")
print("🔍 Verificación de integridad: .tflite vs .h")
for name, cfg in MODELS.items():
    original = cfg["tflite"].read_bytes()
    header_path = MODELS_TFL / f"{cfg['header_name']}.h"
    
    # Extraer bytes del header reconstructivamente (leer el .h y parsear hex)
    # Más simple: verificar tamaño
    original_size = len(original)
    header_content = header_path.read_text()
    declared_size = int(header_content.split("_data_len = ")[1].split(";")[0])
    
    match = original_size == declared_size
    print(f"   {'✅' if match else '❌'} {name}: original={original_size:,} bytes, header declares={declared_size:,} bytes")


📦 MBNTv2_ssdlite_v1
   Input:  [1, 224, 224, 3] (<class 'numpy.int8'>)
   Outputs: 3
   ├─ StatefulPartitionedCall_1:2
   │  Shape: (1, 1470, 1)
   │  Dtype: float32
   │  Range: [0.0000, 0.3359]
   │  Mean:  0.0814
   ├─ StatefulPartitionedCall_1:1
   │  Shape: (1, 1470, 5)
   │  Dtype: float32
   │  Range: [0.0000, 0.5703]
   │  Mean:  0.0539
   ├─ StatefulPartitionedCall_1:0
   │  Shape: (1, 1470, 4)
   │  Dtype: float32
   │  Range: [0.0234, 0.9609]
   │  Mean:  0.4644

📦 YOLO11n_v1
   Input:  [1, 224, 224, 3] (<class 'numpy.float32'>)
   Outputs: 1
   ├─ Identity
   │  Shape: (1, 9, 1029)
   │  Dtype: float32
   │  Range: [0.0000, 0.9835]
   │  Mean:  0.1629

📦 YOLO26n_v1
   Input:  [1, 224, 224, 3] (<class 'numpy.float32'>)
   Outputs: 1
   ├─ Identity
   │  Shape: (1, 300, 6)
   │  Dtype: float32
   │  Range: [-0.0792, 4.0000]
   │  Mean:  0.5898

🔍 Verificación de integridad: .tflite vs .h
   ✅ MBNTv2_ssdlite_v1: original=1,260,680 bytes, header declares=1,260,680 bytes
   ✅ Y

INFO: Applying 1 TensorFlow Lite delegate(s) lazily.
VERBOSE: XNNPack weight cache not enabled.
VERBOSE: Replacing 79 out of 91 node(s) with delegate (TfLiteXNNPackDelegate) node, yielding 3 partitions for the whole graph.
INFO: Successfully applied the default TensorFlow Lite delegate indexed at 0.
 *NOTE*: because a delegate has been applied, the precision of computations should be unchanged, but the exact output tensor values may have changed. If such output values are checked in your code, like in your tests etc., please consider increasing error tolerance for the check.
INFO: Applying 1 TensorFlow Lite delegate(s) lazily.
VERBOSE: XNNPack weight cache not enabled.
VERBOSE: Replacing 315 out of 343 node(s) with delegate (TfLiteXNNPackDelegate) node, yielding 27 partitions for the whole graph.
INFO: Successfully applied the default TensorFlow Lite delegate indexed at 0.
 *NOTE*: because a delegate has been applied, the precision of computations should be unchanged, but the exact out

---
# 5. Informe resumen de conversión

In [12]:
"""
5.1 — Tabla resumen final de modelos convertidos
"""
from IPython.display import Markdown

summary_rows = []
for name, cfg in MODELS.items():
    tfl = cfg["tflite"]
    hdr = MODELS_TFL / f"{cfg['header_name']}.h"
    onnx_p = cfg.get("onnx")
    espdl_p = MODELS_DL / f"{cfg['espdl_name']}.espdl"
    
    summary_rows.append({
        "Modelo": name,
        "Familia": cfg["family"],
        "TFLite INT8 (MB)": f"{tfl.stat().st_size / (1024*1024):.2f}",
        "C Header": "✅ generado" if hdr.exists() else "❌",
        "ONNX": "✅" if (onnx_p and onnx_p.exists()) else "❌",
        "ESPDL": "✅" if espdl_p.exists() else "⏳ pendiente",
        "Output shape": cfg["output_desc"],
        "NMS en device": cfg["nms"],
    })

df_summary = pd.DataFrame(summary_rows).set_index("Modelo")
display(df_summary)

# Estadísticas de archivos generados
print("\n📁 Archivos generados:")
for f in sorted(MODELS_TFL.glob("*.h")):
    print(f"   📄 {f.relative_to(DEPLOY)} ({f.stat().st_size / (1024*1024):.2f} MB)")
    
espdl_files = list(MODELS_DL.glob("*.espdl"))
if espdl_files:
    for f in sorted(espdl_files):
        print(f"   📄 {f.relative_to(DEPLOY)} ({f.stat().st_size / (1024*1024):.2f} MB)")
else:
    print(f"   ⏳ ESPDL: pendiente (ejecutar convert_onnx_to_espdl.py)")

print(f"\n📄 Script de conversión ESPDL: {(DEPLOY / 'convert_onnx_to_espdl.py').relative_to(DEPLOY)}")

# Operadores TFLite Micro
print(f"\n🔧 Operadores TFLite Micro requeridos ({len(all_ops)}):")
for op in sorted(all_ops):
    models_using = [n for n, ops in ops_report.items() if op in ops]
    print(f"   └─ {op:30s} ({', '.join(models_using)})")

# Markdown resumen
display(Markdown(f"""
### ✅ Conversión completada

| Formato | Estado | Archivos | Ubicación |
|---|---|---|---|
| **TFLite → C header** | ✅ Completado | 3 headers (.h) | `main/models/tflite/` |
| **ONNX → ESPDL** | ⏳ Pendiente | Script generado | `convert_onnx_to_espdl.py` |
| **Keras → ONNX** | ✅ Completado | MBNTv2 exportado | `outputs/MBNTv2_ssdlite_v1/` |

**Próximo paso:** Ejecutar `convert_onnx_to_espdl.py` en entorno con `esp-ppq` + PyTorch para 
generar los archivos `.espdl`. Luego proceder con la compilación del firmware ESP-IDF.

**Total en flash (TFLite path):** {sum(M['tflite'].stat().st_size for M in MODELS.values()) / (1024*1024):.2f} MB de 15 MB disponibles → **caben los 3 modelos simultáneamente**.
"""))

,Familia,TFLite INT8 (MB),C Header,ONNX,ESPDL,Output shape,NMS en device
Modelo,,,,,,,
MBNTv2_ssdlite_v1,MobileNetV2,1.20,✅ generado,✅,⏳ pendiente,"3 tensores: bbox[1,1470,4], class[1,1470,5], o...",Decode anchors + NMS
YOLO11n_v1,YOLO11,2.68,✅ generado,✅,⏳ pendiente,"1 tensor: [1,9,1029]",NMS completo en device
YOLO26n_v1,YOLO26,2.55,✅ generado,✅,⏳ pendiente,"1 tensor: [1,300,6]",NMS integrado (no requiere NMS)



📁 Archivos generados:
   📄 main/models/tflite/mobilenetv2_ssdlite_v1_int8.h (7.61 MB)
   📄 main/models/tflite/yolo11n_v1_int8.h (16.99 MB)
   📄 main/models/tflite/yolo26n_v1_int8.h (16.17 MB)
   ⏳ ESPDL: pendiente (ejecutar convert_onnx_to_espdl.py)

📄 Script de conversión ESPDL: convert_onnx_to_espdl.py

🔧 Operadores TFLite Micro requeridos (26):
   └─ ADD                            (MBNTv2_ssdlite_v1, YOLO11n_v1, YOLO26n_v1)
   └─ BATCH_MATMUL                   (YOLO11n_v1, YOLO26n_v1)
   └─ CAST                           (YOLO26n_v1)
   └─ CONCATENATION                  (YOLO11n_v1, YOLO26n_v1)
   └─ CONV_2D                        (MBNTv2_ssdlite_v1, YOLO11n_v1, YOLO26n_v1)
   └─ DEPTHWISE_CONV_2D              (MBNTv2_ssdlite_v1, YOLO11n_v1, YOLO26n_v1)
   └─ DEQUANTIZE                     (MBNTv2_ssdlite_v1)
   └─ FLOOR_MOD                      (YOLO26n_v1)
   └─ GATHER                         (YOLO26n_v1)
   └─ GATHER_ND                      (YOLO26n_v1)
   └─ LESS               


### ✅ Conversión completada

| Formato | Estado | Archivos | Ubicación |
|---|---|---|---|
| **TFLite → C header** | ✅ Completado | 3 headers (.h) | `main/models/tflite/` |
| **ONNX → ESPDL** | ⏳ Pendiente | Script generado | `convert_onnx_to_espdl.py` |
| **Keras → ONNX** | ✅ Completado | MBNTv2 exportado | `outputs/MBNTv2_ssdlite_v1/` |

**Próximo paso:** Ejecutar `convert_onnx_to_espdl.py` en entorno con `esp-ppq` + PyTorch para 
generar los archivos `.espdl`. Luego proceder con la compilación del firmware ESP-IDF.

**Total en flash (TFLite path):** 6.44 MB de 15 MB disponibles → **caben los 3 modelos simultáneamente**.
